# 🔬 Lensless Face Recognition (Transformer Version) — Google Colab
**Paper:** Privacy-Preserving Face Recognition and Verification With Lensless Camera (IEEE TBIOM 2024)
**Implementation:** DCT-ViT (Frequency Domain Vision Transformer)

### ⚡ Before running: Enable GPU
> Runtime → Change runtime type → Hardware accelerator → **T4 GPU** → Save

---

## Step 1: Check GPU
Ensures Colab allocated a T4 GPU for your session.

In [ ]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
else:
    print('WARNING: No GPU found. Go to Runtime > Change runtime type > GPU')

## Step 2: Clone the Repository & Install Dependencies
Clones the codebase containing the Transformer model.

In [ ]:
import os
import sys

if not os.path.exists('Face-Verification-and-identification-using-Lens-less-camera-and-transformer-model'):
    !git clone https://github.com/SatyamSingh-Git/Face-Verification-and-identification-using-Lens-less-camera-and-transformer-model.git
else:
    print('Repo already cloned.')
%cd Face-Verification-and-identification-using-Lens-less-camera-and-transformer-model

!pip install -q joblib scikit-learn scipy six tensorboard
print('Dependencies installed!')

## Step 3: 🚀 Download Dataset directly to Colab
We use Colab's fast internet to download and unzip the 19GB dataset straight to the instance. **(Takes ~12 minutes)**.

*Note: No Google Drive authentication required!*

In [ ]:
import os

# 1. Download and extract the 19GB Dataset directly to Colab
os.makedirs('/content/lensless_data', exist_ok=True)
!wget -O /content/lensless_data.zip "https://mailmissouri-my.sharepoint.com/:u:/g/personal/chffn_umsystem_edu/IQBgLURyOuKgSrwfl8fLn8ipAY6Ikc-va09tctmaHQaVGcY?e=1Xz4Bo&download=1"
!unzip -q /content/lensless_data.zip -d /content/

# 2. Set pointers so the training scripts know where to find the data
TRAIN_DATA = '/content/lensless_data/train/ymdct_npy'
TEST_DATA  = '/content/lensless_data/test/ymdct_npy'
os.environ['TRAIN_DATA'] = TRAIN_DATA
os.environ['TEST_DATA'] = TEST_DATA

print('\nData Extraction complete!')
print('Train data ready:', os.path.exists(TRAIN_DATA))
print('Test data ready: ', os.path.exists(TEST_DATA))

## Step 4: 🤖 Train Transformer Model (DCT-ViT)
Start training the Transformer from scratch. The script automatically uses `AdamW` and `CosineAnnealingLR`.
Best checkpoints will be saved to the `logs/best.pth` directory.

In [ ]:
!python train.py \
    --model transformer \
    --train_data $TRAIN_DATA \
    --test_data  $TEST_DATA \
    --batch_size 32 \
    --lr 1e-4 \
    --num_epoch 100

## Step 5: View Training Progress Log (Live)
Execute this cell to open the TensorBoard interface (even while Step 4 is running) to visualize your training curve.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

## Step 6: 🤖 Test Transformer — Face Recognition (Identification)
Evaluate the Identification accuracy of your trained Transformer model.

> If you have a different checkpoint name, update the `--weights` path.

In [ ]:
!python test_face_recognition.py \
    --model transformer \
    --test_data $TEST_DATA \
    --weights logs/best.pth \
    --batch_size 32

## Step 7: 🤖 Test Transformer — Face Verification (ROC Curve)
Evaluate the Verification AUC and True Positive / False Positive rates. 
This script will generate and display an ROC Curve image (`roc_curve.png`).

In [ ]:
!python test_face_verification.py \
    --model transformer \
    --test_data $TEST_DATA \
    --pairs data/verification_pairs.txt \
    --weights logs/best.pth
